In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings('ignore')

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists():
            return candidate
    return current

root = locate_root()
data_dir = root / 'data'
metadata_dir = data_dir / 'metadata'
processed_dir = data_dir / 'processed'
models_dir = root / 'models'
reports_dir = root / 'reports'
figures_dir = reports_dir / 'figures'
tables_dir = reports_dir / 'tables'

for d in [figures_dir, tables_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Root:', root)

In [ ]:
standardized_df = pd.read_parquet(processed_dir / 'flight_rows_standardized.parquet')
scores_df = pd.read_parquet(processed_dir / 'anomaly_model_scores.parquet')
feature_manifest = pd.read_csv(metadata_dir / 'feature_manifest.csv')
models = joblib.load(models_dir / 'anomaly_models.pkl')

available_features = models['features']
scaler = models['scaler']
pca_pre = models['pca_pre']
pca_recon = models['pca_recon']
iso_refined = models['iso_refined']
ensemble_threshold = models['ensemble_threshold']
spe_threshold_95 = models['spe_threshold_95']
t2_threshold_95 = models['t2_threshold_95']

y_true = (standardized_df['group'] == 'defective').astype(int).values
print(f'Rows: {len(y_true):,}  Defective: {y_true.sum():,} ({y_true.mean():.1%})')

In [ ]:
def evaluate_model(name, y_true, y_pred, y_score):
    return {
        'model': name,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'avg_precision': average_precision_score(y_true, y_score),
        'tp': int(((y_pred == 1) & (y_true == 1)).sum()),
        'fp': int(((y_pred == 1) & (y_true == 0)).sum()),
        'tn': int(((y_pred == 0) & (y_true == 0)).sum()),
        'fn': int(((y_pred == 0) & (y_true == 1)).sum()),
    }

comparisons = [
    evaluate_model(
        'Isolation Forest (original)',
        y_true, scores_df['iso_anomaly'].values, -scores_df['iso_score'].values
    ),
    evaluate_model(
        'Isolation Forest (refined)',
        y_true, scores_df['iso_refined_anomaly'].values, -scores_df['iso_refined_score'].values
    ),
    evaluate_model(
        'One-Class SVM',
        y_true, scores_df['ocsvm_anomaly'].values, -scores_df['ocsvm_score'].values
    ),
    evaluate_model(
        'PCA SPE (95%)',
        y_true, scores_df['spe_anomaly_95'].values, scores_df['spe'].values
    ),
    evaluate_model(
        'PCA T² (95%)',
        y_true, scores_df['t2_anomaly_95'].values, scores_df['t2'].values
    ),
    evaluate_model(
        'Ensemble (refined)',
        y_true, scores_df['ensemble_refined_anomaly'].values, scores_df['ensemble_score_norm'].values
    ),
]

comparison_df = pd.DataFrame(comparisons)
comparison_df.to_csv(tables_dir / 'model_comparison_table.csv', index=False)
print(comparison_df[['model', 'f1', 'precision', 'recall', 'roc_auc', 'avg_precision']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

model_pred_pairs = [
    ('Isolation Forest (original)', scores_df['iso_anomaly'].values),
    ('Isolation Forest (refined)', scores_df['iso_refined_anomaly'].values),
    ('One-Class SVM', scores_df['ocsvm_anomaly'].values),
    ('PCA SPE (95%)', scores_df['spe_anomaly_95'].values),
    ('PCA T² (95%)', scores_df['t2_anomaly_95'].values),
    ('Ensemble (refined)', scores_df['ensemble_refined_anomaly'].values),
]

for ax, (name, y_pred) in zip(axes.flat, model_pred_pairs):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy', 'Defective'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9)

plt.suptitle('Confusion Matrices: All Models', fontsize=12)
plt.tight_layout()
plt.savefig(figures_dir / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['f1', 'precision', 'recall', 'roc_auc', 'avg_precision']
x = np.arange(len(comparisons))
width = 0.15
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [c[metric] for c in comparisons]
    ax.bar(x + i * width, vals, width, label=metric.upper().replace('_', ' '), color=color, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels([c['model'] for c in comparisons], rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(figures_dir / 'model_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_all = standardized_df[available_features].fillna(0).astype(np.float32)
X_all_scaled = scaler.transform(X_all)

X_recon = pca_recon.inverse_transform(pca_recon.transform(X_all_scaled))
residuals = X_all_scaled - X_recon
feature_residuals = pd.Series(
    np.mean(np.abs(residuals), axis=0),
    index=available_features
).sort_values(ascending=False)

healthy_mask = standardized_df['group'] == 'healthy'
defective_mask = standardized_df['group'] == 'defective'

residuals_h = residuals[healthy_mask.values]
residuals_d = residuals[defective_mask.values]

feature_contrast = pd.Series(
    np.mean(np.abs(residuals_d), axis=0) - np.mean(np.abs(residuals_h), axis=0),
    index=available_features
).sort_values(ascending=False)

feature_importance_df = pd.DataFrame({
    'feature': available_features,
    'mean_abs_residual_all': np.mean(np.abs(residuals), axis=0),
    'mean_abs_residual_healthy': np.mean(np.abs(residuals_h), axis=0),
    'mean_abs_residual_defective': np.mean(np.abs(residuals_d), axis=0),
    'defective_minus_healthy': feature_contrast.values
}).sort_values('defective_minus_healthy', ascending=False)

feature_importance_df.to_csv(tables_dir / 'feature_importance_residuals.csv', index=False)
print(feature_importance_df.head(15).to_string(index=False))

In [ ]:
top_n = 15
top_features = feature_importance_df.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_features['feature'][::-1], top_features['defective_minus_healthy'][::-1], color='#E87722', alpha=0.85)
axes[0].set_xlabel('Defective - Healthy Mean |Residual|')
axes[0].set_title('Top Features: Residual Contrast (Defective vs Healthy)')
axes[0].axvline(0, color='black', linewidth=0.8)

x = np.arange(top_n)
w = 0.35
axes[1].barh(x + w/2, top_features['mean_abs_residual_healthy'][::-1].values, w, label='Healthy', color='#4878CF', alpha=0.85)
axes[1].barh(x - w/2, top_features['mean_abs_residual_defective'][::-1].values, w, label='Defective', color='#E87722', alpha=0.85)
axes[1].set_yticks(x)
axes[1].set_yticklabels(top_features['feature'][::-1].values, fontsize=9)
axes[1].set_xlabel('Mean Absolute Residual')
axes[1].set_title('Per-Group Residuals for Top Features')
axes[1].legend()

plt.suptitle('PCA Residual-Based Feature Importance', fontsize=12)
plt.tight_layout()
plt.savefig(figures_dir / 'feature_importance_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_all_scaled_full = scaler.transform(X_all)

sample_size = min(20_000, len(X_all_scaled_full))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_all_scaled_full), sample_size, replace=False)

X_sample = X_all_scaled_full[sample_idx]
groups_sample = standardized_df['group'].values[sample_idx]

pca_2d.fit(X_all_scaled_full)
X_2d = pca_2d.transform(X_sample)

fig, ax = plt.subplots(figsize=(10, 7))
for group, color, label in [('healthy', '#4878CF', 'Healthy'), ('defective', '#E87722', 'Defective')]:
    mask = groups_sample == group
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, alpha=0.3, s=3, label=label)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('PCA 2D Projection: Healthy vs Defective')
ax.legend(markerscale=4)

plt.tight_layout()
plt.savefig(figures_dir / 'pca_2d_separation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
file_validation_summary = scores_df.groupby('file_name').agg(
    group=('group', 'first'),
    total_rows=('sequence_index', 'count'),
    ensemble_anomaly_rate=('ensemble_refined_anomaly', 'mean'),
    spe_anomaly_rate=('spe_anomaly_95', 'mean'),
    iso_anomaly_rate=('iso_refined_anomaly', 'mean'),
    mean_ensemble_score=('ensemble_score_norm', 'mean'),
    mean_spe=('spe', 'mean'),
    p95_spe=('spe', lambda x: np.percentile(x, 95)),
).reset_index()

file_validation_summary['predicted_group'] = np.where(
    file_validation_summary['ensemble_anomaly_rate'] >= 0.3, 'defective', 'healthy'
)
file_validation_summary['correct'] = (
    file_validation_summary['predicted_group'] == file_validation_summary['group']
)

file_accuracy = file_validation_summary['correct'].mean()
print(f'File-level classification accuracy: {file_accuracy:.1%}')
print(file_validation_summary[['file_name', 'group', 'predicted_group', 'correct', 'ensemble_anomaly_rate']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_files = file_validation_summary.sort_values('mean_ensemble_score', ascending=True)
bar_colors = ['#E87722' if g == 'defective' else '#4878CF' for g in sorted_files['group']]
hatch = ['' if c else '///' for c in sorted_files['correct']]

bars = axes[0].barh(sorted_files['file_name'], sorted_files['mean_ensemble_score'], color=bar_colors)
for bar, h in zip(bars, hatch):
    bar.set_hatch(h)
axes[0].axvline(0.3, color='red', linewidth=1.2, linestyle='--', label='Decision threshold (0.30)')
axes[0].set_xlabel('Mean Ensemble Score')
axes[0].set_title(f'File Classification (accuracy={file_accuracy:.1%})')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='y', labelsize=7)

import matplotlib.patches as mpatches
p1 = mpatches.Patch(color='#4878CF', label='Healthy (true)')
p2 = mpatches.Patch(color='#E87722', label='Defective (true)')
p3 = mpatches.Patch(facecolor='white', hatch='///', edgecolor='black', label='Misclassified')
axes[0].legend(handles=[p1, p2, p3], fontsize=8)

axes[1].boxplot(
    [
        file_validation_summary.loc[file_validation_summary['group'] == 'healthy', 'mean_ensemble_score'],
        file_validation_summary.loc[file_validation_summary['group'] == 'defective', 'mean_ensemble_score'],
    ],
    labels=['Healthy', 'Defective'],
    patch_artist=True,
    boxprops=dict(facecolor='#aec7e8'),
    medianprops=dict(color='black', linewidth=2),
    notch=False
)
axes[1].set_ylabel('Mean Ensemble Score')
axes[1].set_title('Score Distribution by True Group')

plt.suptitle('File-Level Validation', fontsize=12)
plt.tight_layout()
plt.savefig(figures_dir / 'file_level_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_defective_files = file_validation_summary[
    file_validation_summary['group'] == 'defective'
].nlargest(4, 'mean_ensemble_score')['file_name'].tolist()

fig, axes = plt.subplots(len(top_defective_files), 1, figsize=(14, 4 * len(top_defective_files)))
if len(top_defective_files) == 1:
    axes = [axes]

for ax, fname in zip(axes, top_defective_files):
    fdata = scores_df[scores_df['file_name'] == fname].sort_values('sequence_index')
    ax.fill_between(fdata['sequence_index'], fdata['ensemble_score_norm'], alpha=0.5, color='#E87722', label='Ensemble Score')
    ax.axhline(ensemble_threshold, color='red', linewidth=1.2, linestyle='--', label=f'Threshold={ensemble_threshold:.3f}')
    ax.set_title(fname, fontsize=10)
    ax.set_xlabel('Sequence Index')
    ax.set_ylabel('Ensemble Score')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('Temporal Anomaly Profiles: Top Defective Files', fontsize=12)
plt.tight_layout()
plt.savefig(figures_dir / 'temporal_anomaly_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
final_summary = {
    'total_rows': len(y_true),
    'healthy_rows': int((y_true == 0).sum()),
    'defective_rows': int(y_true.sum()),
    'best_model': comparison_df.loc[comparison_df['f1'].idxmax(), 'model'],
    'best_f1': float(comparison_df['f1'].max()),
    'best_roc_auc': float(comparison_df['roc_auc'].max()),
    'file_level_accuracy': float(file_accuracy),
    'top_features': feature_importance_df.head(5)['feature'].tolist(),
}

pd.DataFrame([final_summary]).to_csv(tables_dir / 'validation_final_summary.csv', index=False)
file_validation_summary.to_csv(tables_dir / 'file_level_validation.csv', index=False)

print('=== VALIDATION SUMMARY ===')
for k, v in final_summary.items():
    print(f'  {k}: {v}')